# 01_EDA: Exploratory Data Analysis - Temporal Drift Study

This notebook performs exploratory data analysis on temporal datasets, setting up the foundation for drift detection analysis. We load datasets, explore temporal structure, and prepare data for downstream drift analysis.

## Research Questions:
- RQ1: Which features are most susceptible to temporal drift?
- RQ2: Is there correlation between feature drift and model performance?
- RQ3: At what drift threshold does performance degrade?
- RQ4: Which retraining strategy is optimal?


In [ ]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add src to path
sys.path.insert(0, '../src')

# Set seeds for reproducibility
np.random.seed(42)

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# Load configuration
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully")
print(f"Dataset: {config['dataset']['name']}")
print(f"Number of windows: {config['time_window']['n_windows']}")


## 1. Load and Generate Temporal Data


In [ ]:
from data_loader import DataLoader, SyntheticDriftDataset

# Initialize data loader
loader = DataLoader(random_state=42)

# Load synthetic data with temporal drift
df_synthetic, window_ids = loader.load_synthetic_data(
    n_samples_per_window=config['dataset']['synthetic']['n_samples_per_window'],
    n_features=config['dataset']['synthetic']['n_features'],
    n_windows=config['dataset']['synthetic']['n_windows'],
    drift_magnitude=config['dataset']['synthetic']['drift_magnitude'],
    drift_type=config['dataset']['synthetic']['drift_type']
)

print(f"Synthetic dataset shape: {df_synthetic.shape}")
print(f"\nFirst few rows:")
print(df_synthetic.head())
print(f"\nData types:")
print(df_synthetic.dtypes)
print(f"\nTarget class distribution:")
print(df_synthetic['target'].value_counts())


## 2. Split Data into Time Windows


In [ ]:
# Split into time windows
n_windows = config['time_window']['n_windows']
windows = loader.split_into_windows(df_synthetic, n_windows=n_windows)

print(f"Split into {len(windows)} windows")
print(f"Each window has approximately {len(windows[0][0])} samples")

# Create ref-test splits
splits = loader.get_ref_and_test_splits(windows, ref_window_idx=0)

# Standardize features
splits_standardized = loader.standardize_splits(splits)

print(f"\nCreated {len(splits_standardized)} ref-test splits")
print(f"Reference window shape: {splits_standardized[0]['X_ref'].shape}")
print(f"Test window shape: {splits_standardized[0]['X_test'].shape}")


## 3. Explore Feature Statistics and Distributions


In [ ]:
# Compute feature statistics across windows
feature_names = splits_standardized[0]['feature_names']
n_features = len(feature_names)

# Calculate mean and std for each feature across windows
window_means = []
window_stds = []

for split in splits_standardized:
    X_test = split['X_test']
    window_means.append(X_test.mean(axis=0))
    window_stds.append(X_test.std(axis=0))

window_means = np.array(window_means)
window_stds = np.array(window_stds)

print("Feature Statistics Summary:")
print(f"Mean shape: {window_means.shape} (windows × features)")
print(f"Std shape: {window_stds.shape} (windows × features)")

# Create summary statistics DataFrame
stats_df = pd.DataFrame({
    'feature': feature_names,
    'mean_across_windows': window_means.mean(axis=0),
    'std_across_windows': window_means.std(axis=0),
    'mean_std': window_stds.mean(axis=0),
})

print("\nFeature Summary Statistics:")
print(stats_df)


## 4. Visualize Feature Distributions Over Time


In [ ]:
# Plot feature means over time
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i in range(min(4, n_features)):
    ax = axes[i]
    ax.plot(range(len(window_means)), window_means[:, i], marker='o', linewidth=2, markersize=6)
    ax.fill_between(
        range(len(window_means)),
        window_means[:, i] - window_stds[:, i],
        window_means[:, i] + window_stds[:, i],
        alpha=0.3
    )
    ax.set_xlabel("Time Window")
    ax.set_ylabel("Mean Value (standardized)")
    ax.set_title(f"{feature_names[i]}: Mean Over Time")
    ax.grid(True, alpha=0.3)

plt.suptitle("Feature Mean Values Across Time Windows", fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('../reports/figures/01_feature_means_over_time.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved feature means visualization")


## 5. Summary and Next Steps

**EDA Findings:**
- Dataset: Synthetic temporal drift with {n_windows} time windows
- Features: {n_features} standardized features
- Samples per window: {samples_per_window}
- Target: Binary classification

**Observations:**
- Some features show clear drift patterns over time
- Feature means and variances change across windows
- Dataset is suitable for drift analysis

**Next Steps:**
- **Notebook 02**: Implement and compute drift detection metrics (PSI, KL, KS test)
- **Notebook 03**: Train baseline models and track performance degradation
- **Notebook 04**: Implement and compare retraining strategies
- **Notebook 05**: Visualize results and answer research questions
